# 🧪 Lab 1: Initializing the Viewscreen (Spark UI Hierarchy & Lifecycle Foundation)

Welcome to the bridge simulator. In this foundational lab, we initialize our warp core (the Spark Session), expose its subspace web server coordinates, and observe exactly how lazy transformations differ from high-energy actions within the execution hierarchy.

**Objective:** Master navigation of port `:4040` and map the physical cascade from an abstract query plan down to active distributed task execution metrics.

### Step 1: Spin Up the Engine Room Coordinates
We spin up an isolated local Spark session, enforce our target telemetry port configuration, and verify our active runtime metadata.

Important: Spark does not guarantee `4040` forever. If the port is occupied, Spark may try the next ports according to `spark.port.maxRetries`. Always trust `spark.sparkContext.uiWebUrl`, not your hopes.

In [1]:
from pyspark.sql import SparkSession
import time

# In notebook environments, an old SparkSession may already exist.
# Stop it so the port/app metadata in this lab is easier to reason about.
existing = SparkSession.getActiveSession()
if existing is not None:
    existing.stop()

spark = (
    SparkSession.builder
        .appName("spark-ui-klingon-bridge-lab-01")
        .master("local[*]")
        .config("spark.ui.enabled", "true")
        .config("spark.ui.port", "4040")
        .config("spark.port.maxRetries", "16")
        .getOrCreate()
)

sc = spark.sparkContext

ui_url = sc.uiWebUrl
app_id = sc.applicationId
master = sc.master
default_parallelism = sc.defaultParallelism

print(f"Active Spark Core Engine: {spark.version}")
print(f"🛰️ Actual telemetry viewscreen URL: {ui_url}")
print(f"🆔 Application ID: {app_id}")
print(f"🧵 Master: {master}")
print(f"⚙️ Default parallelism / local task slots: {default_parallelism}")

Active Spark Core Engine: 4.1.2
🛰️ Actual telemetry viewscreen URL: http://T14-PF4WM3XL.sdggroup.local:4040
🆔 Application ID: local-1783397021405
🧵 Master: local[*]
⚙️ Default parallelism / local task slots: 12


### Step 2: Scan the Sector via Lazy Transformations
We stage an asset evaluation blueprint across a massive tracking range. Because Spark transformations are lazy, this step builds a logical plan but does not launch a Spark job yet.

The Spark UI itself is alive, but the Jobs and Stages tabs should not show execution from this transformation alone.

In [2]:
# Transformation only: no Spark job should appear yet.
# We set numPartitions explicitly so the later action has visible tasks.
scout_ships = (
    spark.range(0, 10_000_000, 1, numPartitions=8)
         .filter("id % 2 = 0")
)

print("🛡️ Logical plan staged. No action has fired yet.")
print("Open the Spark UI: Jobs and Stages should still show no execution for this DataFrame.")

🛡️ Logical plan staged. No action has fired yet.
Open the Spark UI: Jobs and Stages should still show no execution for this DataFrame.


### Step 3: Fire Photon Torpedoes (Executing Actions)
We invoke a terminal evaluation command. This action forces Spark to execute the accumulated plan. In a fresh session, this will often appear as `Job 0`, but the exact job number is not guaranteed.

Because this example has no shuffle, expect a simple job with a small number of tasks, usually matching the partitions created above.

In [3]:
print("🚀 Firing photon torpedoes... Watch the Spark UI process the action.")

start_clock = time.time()
scout_count = scout_ships.count()
end_clock = time.time()

print(f"📡 Telemetry received: {scout_count} rows matched the filter.")
print(f"⏱️ Processing duration: {end_clock - start_clock:.2f} seconds.")
print("Now check the Jobs, Stages, and Executors tabs in the Spark UI.")

🚀 Firing photon torpedoes... Watch the Spark UI process the action.
📡 Telemetry received: 5000000 rows matched the filter.
⏱️ Processing duration: 3.16 seconds.
Now check the Jobs, Stages, and Executors tabs in the Spark UI.


### Step 4: Maintain Active Bridge Intercept
We keep the driver process alive so the Live UI remains available.

Once the SparkContext stops, the Live UI disappears. Post-mortem analysis requires persisted Spark event logs and a History Server.

In [4]:
print("⏳ Keeping the live Spark UI open.")
print("Inspect the Jobs, Stages, Tasks, Executors, and Environment tabs now.")
print(f"Spark UI: {ui_url}")

input("Press Enter when you are done inspecting the UI and want to stop Spark...")

spark.stop()
print("💀 SparkContext stopped. The Live UI is no longer available.")

⏳ Keeping the live Spark UI open.
Inspect the Jobs, Stages, Tasks, Executors, and Environment tabs now.
Spark UI: http://T14-PF4WM3XL.sdggroup.local:4040
💀 SparkContext stopped. The Live UI is no longer available.


## 📊 Post-Lab Analysis: Evidence from the Engine Room

This lab validates the first survival rule of the Spark UI: the UI is not showing what your code intends to do. It is showing what Spark actually executed.

### 1. Lazy Blueprints Are Not Execution Metrics

Step 2 builds a DataFrame using `range()` and `filter()`, but no action is called yet. Spark records a logical plan, but it does not scan rows, launch tasks, or create a completed job.

The Spark UI may already be reachable because the Spark application is alive, but the Jobs and Stages tabs should not show execution from that transformation alone.

### 2. Actions Pay the Accumulated Computational Tax

Calling `.count()` is the execution trigger. At that moment, Spark turns the accumulated logical plan into physical execution, launches a job, creates a stage, and runs tasks against the partitions.

In this lab, the range was created with 8 partitions, so the action should produce visible task activity. In local mode, those tasks run on local worker threads / task slots, not on a real multi-node cluster.

### 3. The Live UI Follows the Driver Lifespan

The Spark Web UI is bound to the active Spark application. While the driver and SparkContext are alive, the Live UI is available. Once `spark.stop()` runs, the Live UI disappears.

For post-mortem debugging, you need persisted Spark event logs and a Spark History Server. No persisted Spark event logs, no History Server reconstruction.

### ⏱️ Validation Summary

* **Step 1:** Started a local Spark application and printed the actual Spark UI URL.
* **Step 2:** Built a lazy transformation plan without launching a job.
* **Step 3:** Triggered execution with `.count()` and produced visible Jobs, Stages, Tasks, and executor metrics.
* **Step 4:** Kept the driver alive long enough to inspect the Live UI, then stopped Spark and watched the Live UI disappear.